#### Random Forest Classifier for Jailbreak Prediction

A Random Forest classifier with hyperparameter tuning using `GridSearchCV`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import logging
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True # ensure logging works correctly in jupyter
)
logger = logging.getLogger(__name__)

##### Random Forest Classifer Class
Has the model logic, including hyperparameter, grid search training, eval metrics, and visualization generation.

In [ ]:
class RandomForestJailbreakClassifier:
    """
    Random Forest classifier with hyperparameter tuning for jailbreak prediction
    """
    
    def __init__(self, random_state=42):
        """
        Init Random Forest classifier
        """
        self.random_state = random_state # for reproducability
        self.model = None
        self.best_params = None
        self.cv_results = None
        self.feature_names = None
        self.results = {}
        
    def define_param_grid(self):
        """
        Define hyperparameter search space. Returns dict of hyperparameters to search
        """
        param_grid = {
            'n_estimators': [100, 200, 300, 500],  # Number of trees
            'max_depth': [10, 20, 30, None],       # Maximum tree depth
            'min_samples_split': [2, 5, 10],       # Min samples to split node
            'min_samples_leaf': [1, 2, 4],         # Min samples in leaf
            'max_features': ['sqrt', 'log2'],      # Features per split
            'bootstrap': [True],                   # Use bootstrap sampling
            'class_weight': [
                'balanced',  # Inversely proportional to class frequencies
                {0: 1, 1: 50},  # penalize jailbreak misses 50x more
                {0: 1, 1: 100}, # more agressive
                'balanced_subsample'
            ]
        }
        
        total_combinations = (
            len(param_grid['n_estimators']) *
            len(param_grid['max_depth']) *
            len(param_grid['min_samples_split']) *
            len(param_grid['min_samples_leaf']) *
            len(param_grid['max_features']) *
            len(param_grid['bootstrap']) *
            len(param_grid['class_weight'])
        )
        
        logger.info(f"Hyperparameter search space: {total_combinations} combinations")
        logger.info(f"Grid: {param_grid}")
        
        return param_grid
    
    def train_with_grid_search(self, X_train, y_train, X_val=None, y_val=None, feature_names=None):
        """
        Train Random Forest with GridSearchCV for hyperparameter tuning
        """
        logger.info("="*60)
        logger.info("RANDOM FOREST: HYPERPARAMETER TUNING")
        logger.info("="*60)
        logger.info(f"Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features")
        
        if feature_names is not None:
            self.feature_names = feature_names
        elif isinstance(X_train, pd.DataFrame):
            self.feature_names = X_train.columns.tolist()
        
        # Define parameter grid
        param_grid = self.define_param_grid()
        
        # Configure cross-validation
        cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_state)
        
        # Initialize base model
        base_model = RandomForestClassifier(
            random_state=self.random_state,
            n_jobs=-1,  # Use all CPU cores
            verbose=0
        )
        
        # Grid search with cross-validation
        logger.info("\nStarting GridSearchCV (this may take a while)...")
        logger.info("Using 5-fold stratified cross-validation")
        logger.info("Scoring metric: recall")
        
        start_time = time.time()
        
        grid_search = GridSearchCV(
            estimator=base_model,
            param_grid=param_grid,
            cv=cv_strategy,
            scoring='recall',          # Optimize recall as described in paper
            n_jobs=-1,                 # Parallel processing
            verbose=2,                 # Progress updates
            return_train_score=True    # Track overfitting
        )
        
        # Fit grid search
        grid_search.fit(X_train, y_train)
        
        elapsed_time = time.time() - start_time
        logger.info(f"\nGrid search complete. Time elapsed: {elapsed_time/60:.2f} minutes")
        
        # Store results
        self.model = grid_search.best_estimator_
        self.best_params = grid_search.best_params_
        self.cv_results = grid_search.cv_results_
        
        # Log best parameters
        logger.info("\n" + "="*60)
        logger.info("BEST HYPERPARAMETERS")
        logger.info("="*60)
        for param, value in self.best_params.items():
            logger.info(f"  {param}: {value}")
        logger.info(f"\nBest CV F1-Score: {grid_search.best_score_:.4f}")
        
        # Evaluate on training set
        y_train_pred = self.model.predict(X_train)
        y_train_proba = self.model.predict_proba(X_train)[:, 1]
        train_f1 = f1_score(y_train, y_train_pred)
        train_auc = roc_auc_score(y_train, y_train_proba)
        
        logger.info(f"\nTraining Set Performance:")
        logger.info(f"  F1-Score: {train_f1:.4f}")
        logger.info(f"  AUC-ROC:  {train_auc:.4f}")
        
        # Evaluate on validation set if provided
        if X_val is not None and y_val is not None:
            val_metrics = self.evaluate(X_val, y_val, 'Validation')
        
        return self.model
    
    def evaluate(self, X, y, dataset_name='Test'):
        """
        Evaluate model performance. Returns dict of eval metrics
        """
        logger.info(f"\nEvaluating on {dataset_name} set...")
        
        # Predictions
        y_pred = self.model.predict(X)
        y_proba = self.model.predict_proba(X)[:, 1]
        
        # Calculate metrics
        metrics = {
            'accuracy': accuracy_score(y, y_pred),
            'precision': precision_score(y, y_pred, zero_division=0),
            'recall': recall_score(y, y_pred, zero_division=0),
            'f1': f1_score(y, y_pred, zero_division=0),
            'auc_roc': roc_auc_score(y, y_proba),
            'confusion_matrix': confusion_matrix(y, y_pred)
        }
        
        # Log metrics
        logger.info(f"{dataset_name} Metrics:")
        logger.info(f"  Accuracy:  {metrics['accuracy']:.4f}")
        logger.info(f"  Precision: {metrics['precision']:.4f}")
        logger.info(f"  Recall:    {metrics['recall']:.4f}")
        logger.info(f"  F1-Score:  {metrics['f1']:.4f}")
        logger.info(f"  AUC-ROC:   {metrics['auc_roc']:.4f}")
        
        # Store results
        self.results[dataset_name.lower()] = metrics
        
        return metrics
    
    def extract_feature_importance(self, top_n=20):
        """
        Extract feature importance based on Gini impurity decrease. Returns df with feature importances
        """
        logger.info("\nExtracting feature importance...")
        
        if self.feature_names is None:
            logger.warning("Feature names not available")
            return None
        
        # Extract importances from all trees
        importances = self.model.feature_importances_
        
        # Calculate standard deviation across trees
        std = np.std([tree.feature_importances_ for tree in self.model.estimators_], axis=0)
        
        # Create importance DataFrame
        importance_df = pd.DataFrame({
            'feature': self.feature_names,
            'importance': importances,
            'std': std
        }).sort_values('importance', ascending=False)
        
        logger.info(f"\nTop {top_n} Most Important Features:")
        logger.info(importance_df.head(top_n).to_string())
        
        return importance_df
    
    def visualize_feature_importance(self, importance_df, top_n=20, save_path='results/rf_feature_importance.png'):
        """
        Visualize feature importance with error bars
        """
        logger.info(f"\nCreating feature importance visualization...")
        
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        
        # Get top features
        top_features = importance_df.head(top_n)
        
        # Create figure
        plt.figure(figsize=(10, 8))
        plt.barh(range(len(top_features)), top_features['importance'], 
                xerr=top_features['std'], alpha=0.7, color='forestgreen')
        plt.yticks(range(len(top_features)), top_features['feature'])
        plt.xlabel('Feature Importance (Gini Decrease)', fontsize=12)
        plt.ylabel('Feature', fontsize=12)
        plt.title(f'Random Forest: Top {top_n} Feature Importances', fontsize=14, fontweight='bold')
        plt.grid(axis='x', alpha=0.3)
        plt.tight_layout()
        
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show() # Added for Notebook display
        logger.info(f"Feature importance plot saved: {save_path}")
        plt.close()
    
    def plot_confusion_matrix(self, y_true, y_pred, save_path='results/rf_confusion_matrix.png'):
        """
        Plot confusion matrix heatmap
        """
        logger.info("Creating confusion matrix visualization...")
        
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        
        cm = confusion_matrix(y_true, y_pred)
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', cbar=True,
                    xticklabels=['Non-Jailbreak', 'Jailbreak'],
                    yticklabels=['Non-Jailbreak', 'Jailbreak'])
        plt.xlabel('Predicted Label', fontsize=12)
        plt.ylabel('True Label', fontsize=12)
        plt.title('Random Forest: Confusion Matrix', fontsize=14, fontweight='bold')
        plt.tight_layout()
        
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show() # Added for Notebook display
        logger.info(f"Confusion matrix saved: {save_path}")
        plt.close()
    
    def plot_roc_curve(self, y_true, y_proba, save_path='results/rf_roc_curve.png'):
        """
        Plot ROC curve
        """
        logger.info("Creating ROC curve...")
        
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        auc = roc_auc_score(y_true, y_proba)
        
        plt.figure(figsize=(8, 6))
        plt.plot(fpr, tpr, linewidth=2, label=f'Random Forest (AUC = {auc:.4f})', color='forestgreen')
        plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
        plt.xlabel('False Positive Rate', fontsize=12)
        plt.ylabel('True Positive Rate', fontsize=12)
        plt.title('ROC Curve: Random Forest', fontsize=14, fontweight='bold')
        plt.legend(loc='lower right', fontsize=10)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.show() # Added for Notebook display
        logger.info(f"ROC curve saved: {save_path}")
        plt.close()
    
    def save_model(self, save_path='models/random_forest_model.pkl'):
        """
        Save trained model and hyperparameters
        """
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        joblib.dump(self.model, save_path)
        logger.info(f"Model saved: {save_path}")
        
        # Save hyperparameters
        params_path = save_path.replace('.pkl', '_params.pkl')
        joblib.dump(self.best_params, params_path)
        logger.info(f"Best parameters saved: {params_path}")
        
        # Save results
        results_path = save_path.replace('.pkl', '_results.pkl')
        joblib.dump(self.results, results_path)
        logger.info(f"Results saved: {results_path}")

##### Data Loading

In [ ]:
def load_preprocessed_data():
    """
    Load preprocessed data (unscaled for Random Forest). returns dict with train/val/test splits
    """
    logger.info("Loading preprocessed data (unscaled for Random Forest)...")
    
    # Load unscaled data (RF doesn't need scaling)
    # Ensure these files exist in your directory
    X_train = pd.read_csv('preprocessed_data/X_train.csv')
    X_val = pd.read_csv('preprocessed_data/X_val.csv')
    X_test = pd.read_csv('preprocessed_data/X_test.csv')
    
    y_train = pd.read_csv('preprocessed_data/y_train.csv')['jailbroken']
    y_val = pd.read_csv('preprocessed_data/y_val.csv')['jailbroken']
    y_test = pd.read_csv('preprocessed_data/y_test.csv')['jailbroken']
    
    metadata = joblib.load('preprocessed_data/metadata.pkl')
    
    logger.info(f"Data loaded successfully")
    logger.info(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    
    return {
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test,
        'metadata': metadata
    }

##### Traning Pipeline

In [ ]:
def train_and_evaluate_random_forest():
    """
    Complete pipeline to train and evaluate Random Forest model. returns trained classifier object
    """
    logger.info("="*60)
    logger.info("RANDOM FOREST: JAILBREAK PREDICTION WITH HYPERPARAMETER TUNING")
    logger.info("="*60)
    
    # Load data
    data = load_preprocessed_data()
    
    # Initialize classifier
    classifier = RandomForestJailbreakClassifier(random_state=42)
    
    # Train with grid search
    classifier.train_with_grid_search(
        data['X_train'],
        data['y_train'],
        data['X_val'],
        data['y_val'],
        feature_names=data['X_train'].columns.tolist()
    )
    
    # Evaluate on test set
    test_metrics = classifier.evaluate(data['X_test'], data['y_test'], 'Test')
    
    # Extract and visualize feature importance
    importance_df = classifier.extract_feature_importance(top_n=20)
    if importance_df is not None:
        classifier.visualize_feature_importance(importance_df, top_n=20)
    
    # Generate visualizations
    y_test_pred = classifier.model.predict(data['X_test'])
    y_test_proba = classifier.model.predict_proba(data['X_test'])[:, 1]
    
    classifier.plot_confusion_matrix(data['y_test'], y_test_pred)
    classifier.plot_roc_curve(data['y_test'], y_test_proba)
    
    # Save model
    classifier.save_model()
    
    # Generate classification report
    logger.info("\nDetailed Classification Report (Test Set):")
    logger.info("\n" + classification_report(data['y_test'], y_test_pred,
                                            target_names=['Non-Jailbreak', 'Jailbreak']))
    
    logger.info("\n" + "="*60)
    logger.info("RANDOM FOREST TRAINING COMPLETE")
    logger.info("="*60)
    
    return classifier

Run the cell below to start training process :)

In [ ]:
# Train and evaluate model
classifier = train_and_evaluate_random_forest()

print("\n" + "="*60)
print("RANDOM FOREST RESULTS SUMMARY")
print("="*60)
print("\nBest Hyperparameters:")
for param, value in classifier.best_params.items():
    print(f"  {param}: {value}")
print(f"\nTest Performance:")
print(f"  Accuracy:  {classifier.results['test']['accuracy']:.4f}")
print(f"  Precision: {classifier.results['test']['precision']:.4f}")
print(f"  Recall:    {classifier.results['test']['recall']:.4f}")
print(f"  F1-Score:  {classifier.results['test']['f1']:.4f}")
print(f"  AUC-ROC:   {classifier.results['test']['auc_roc']:.4f}")
print("\nModel and results saved in 'models/' directory")
print("Visualizations saved in 'results/' directory")
print("="*60)